# Experiment: Output Viewer

Objective:
- Pick an experiment and model.
- Inspect the actual model outputs for that slice.
- Keep the notebook lightweight and easy to rerun.


In [ ]:
from __future__ import annotations

import error_analysis as ea
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

specs_by_dataset = {spec.dataset: spec for spec in ea.STANDARD_EXPERIMENTS}
sorted(specs_by_dataset)

## Controls

Edit the values below, then rerun the notebook from this cell downward.


In [ ]:
ALL_ERROR_TYPES = [
    "exact_match",
    "word_order_only",
    "repetition_loop",
    "copied_source",
    "source_lexicon_intrusion",
    "wrong_script",
    "diacritic_drop",
    "too_short",
    "too_long",
    "hallucinated_vocab",
    "partial_span",
    "same_length_substitution",
    "mixed_other",
    "no_answer",
]

print("Available ERROR_TYPE values:")
for error_type in ALL_ERROR_TYPES:
    print(f"- {error_type}")

In [ ]:
EXPERIMENT = "orthography_large_exp"
MODEL = "gpt-5-nano"
ERROR_TYPE = (
    "diacritic_drop"  # e.g. "wrong_script", "hallucinated_vocab", "word_order_only"
)
N_ROWS = 25
SORT_BY = ["depth", "n_words", "custom_id"]
SHOW_ONLY_ERRORS = False

EXPERIMENT, MODEL, ERROR_TYPE, N_ROWS, SHOW_ONLY_ERRORS

In [ ]:
if EXPERIMENT not in specs_by_dataset:
    raise ValueError(
        f"Unknown experiment: {EXPERIMENT}. Available: {sorted(specs_by_dataset)}"
    )

spec = specs_by_dataset[EXPERIMENT]
df = ea.load_standard_experiment(spec).copy()
df = ea.enrich(df)
df = df[df["fuzzy_model"] == MODEL].copy()

if SHOW_ONLY_ERRORS:
    df = df[~df["exact_match"]].copy()

if ERROR_TYPE is not None:
    df = df[df["failure_mode"] == ERROR_TYPE].copy()

for column in [
    "depth",
    "n_words",
    "n_rules",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

sort_columns = [column for column in SORT_BY if column in df.columns]
if sort_columns:
    df = df.sort_values(sort_columns, na_position="last").reset_index(drop=True)

df["is_correct"] = df["exact_match"]
df["response_preview"] = df["model_response"].fillna("").str.slice(0, 300)

label = f"### Loaded `{len(df):,}` rows for `{MODEL}` on `{EXPERIMENT}`"
if ERROR_TYPE is not None:
    label += f" filtered to error type `{ERROR_TYPE}`"
display(Markdown(label))
df.head(3)

In [ ]:
summary = {
    "experiment": EXPERIMENT,
    "model": MODEL,
    "error_type": ERROR_TYPE,
    "rows": int(len(df)),
    "accuracy": float(df["is_correct"].mean()) if len(df) else float("nan"),
    "mean_prompt_tokens": float(df["prompt_tokens"].mean())
    if len(df)
    else float("nan"),
    "mean_completion_tokens": float(df["completion_tokens"].mean())
    if len(df)
    else float("nan"),
}
pd.Series(summary)

In [ ]:
failure_counts = (
    df["failure_mode"]
    .value_counts(dropna=False)
    .rename_axis("failure_mode")
    .reset_index(name="rows")
)
display(failure_counts)

columns = [
    "custom_id",
    "grammar_name",
    "failure_mode",
    "depth",
    "n_words",
    "n_rules",
    "input_sentence",
    "output_sentence",
    "model_answer",
    "is_correct",
    "prompt_tokens",
    "completion_tokens",
    "response_preview",
]
available_columns = [column for column in columns if column in df.columns]
viewer_df = df[available_columns].head(N_ROWS).copy()
viewer_df

In [ ]:
error_columns = [
    "custom_id",
    "grammar_name",
    "failure_mode",
    "depth",
    "input_sentence",
    "output_sentence",
    "model_answer",
    "response_preview",
]
available_error_columns = [column for column in error_columns if column in df.columns]
df.loc[~df["is_correct"], available_error_columns].head(N_ROWS)